<center><h1>Xia_Iris_HW3</h1></center>
<br>
<br>

Name: Iris Xia
<br>
Github Username: xd11997
<br>
USC ID: 1152659547

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np

Get the AReM Data Set

In [4]:
from pathlib import Path

data_dir = Path.cwd() / "AReM"
columns = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
all_data = {}

for file_path in sorted(data_dir.glob("*/*.csv")):
    activity = file_path.parent.name
    dataset_number = int(file_path.stem.removeprefix("dataset"))
    recording = pd.read_csv(
        file_path,
        sep=r",\s*|\s+",
        engine="python",
        comment="#",
        header=None,
        names=columns,
        usecols=range(len(columns)),
    )
    all_data[(activity, dataset_number)] = recording

print(f"Loaded {len(all_data)} recordings")

Loaded 88 recordings


### (b) Test and Train Data

In [5]:
test_dataset_ids = {"bending1": {1, 2}, "bending2": {1, 2}}

train_data = {}
test_data = {}

for (activity, dataset_number), recording in all_data.items():
    if dataset_number in test_dataset_ids.get(activity, {1, 2, 3}):
        test_data[(activity, dataset_number)] = recording
    else:
        train_data[(activity, dataset_number)] = recording

print(f"Training files: {len(train_data)}")
print(f"Test files: {len(test_data)}")

Training files: 69
Test files: 19


### (c) Feature Extraction

#### i. Research

Time-domain features summarize a signal directly in the time domain. Common examples include:
- Central tendency and quantiles: mean, median, and quantiles such as Q1 and Q3
- Extrema and range: minimum, maximum, and range
- Variability: variance, standard deviation, interquartile range (IQR), and mean absolute deviation
- Distribution shape: skewness and kurtosis
- Amplitude and energy: root mean square (RMS) and signal energy (the sum of squared values)
- Temporal behavior: linear trend or slope, successive-value changes (such as mean absolute change), zero- or threshold-crossing rate, peak count, and autocorrelation at selected lags
Together, these features describe a signal’s typical level, spread, distribution shape, magnitude, and temporal behavior. For multichannel sensor data, they can be calculated separately for each channel in each recording. Frequency-domain measures, such as Fourier coefficients, belong to a different feature family and are not included here.

#### ii. Extraction

In [6]:
feature_rows = {}

for (activity, dataset_number), recording in all_data.items():
    row = {}
    sensor_columns = recording.columns[1:]

    for series_number, column in enumerate(sensor_columns, start=1):
        values = recording[column]
        row[f"minimum_{series_number}"] = values.min()
        row[f"maximum_{series_number}"] = values.max()
        row[f"mean_{series_number}"] = values.mean()
        row[f"median_{series_number}"] = values.median()
        row[f"standard_deviation_{series_number}"] = values.std(ddof=1)
        row[f"first_quartile_{series_number}"] = values.quantile(0.25)
        row[f"third_quartile_{series_number}"] = values.quantile(0.75)

    feature_rows[(activity, dataset_number)] = row

feature_table = pd.DataFrame.from_dict(feature_rows, orient="index")
feature_table.index = pd.MultiIndex.from_tuples(
    feature_table.index, names=["activity", "dataset"]
)

train_features = feature_table.loc[list(train_data)]
test_features = feature_table.loc[list(test_data)]
train_labels = pd.Series(
    train_features.index.get_level_values("activity"),
    index=train_features.index,
    name="activity",
)
test_labels = pd.Series(
    test_features.index.get_level_values("activity"),
    index=test_features.index,
    name="activity",
)

assert feature_table.shape == (88, 42)
assert train_features.shape == (69, 42)
assert test_features.shape == (19, 42)

print(f"All features: {feature_table.shape}")
print(f"Training features: {train_features.shape}")
print(f"Test features: {test_features.shape}")
feature_table.head()

All features: (88, 42)
Training features: (69, 42)
Test features: (19, 42)


minimum_1  maximum_1     mean_1  median_1  \
activity dataset                                              
bending1 1            37.25      45.00  40.624792     40.50   
         2            38.00      45.67  42.812812     42.50   
         3            35.00      47.40  43.954500     44.33   
         4            33.00      47.75  42.179812     43.50   
         5            33.00      45.75  41.678063     41.75   

                  standard_deviation_1  first_quartile_1  third_quartile_1  \
activity dataset                                                             
bending1 1                    1.476967             39.25             42.00   
         2                    1.435550             42.00             43.67   
         3                    1.558835             43.00             45.00   
         4                    3.670666             39.15             45.00   
         5                    2.243490             41.33             42.75   

                  minimum_2  maximum_2    mean_2  ...  standard_deviation_5  \
activity dataset                                  ...                         
bending1 1              0.0       1.30  0.358604  ...              2.188449   
         2              0.0       1.22  0.372437  ...              1.995255   
         3              0.0       1.70  0.426250  ...              1.999604   
         4              0.0       3.00  0.696042  ...              3.849448   
         5              0.0       2.83  0.535979  ...              2.411026   

                  first_quartile_5  third_quartile_5  minimum_6  maximum_6  \
activity dataset                                                             
bending1 1                 33.0000             36.00        0.0       1.92   
         2                 32.0000             34.50        0.0       3.11   
         3                 35.3625             36.50        0.0       1.79   
         4                 30.4575             36.33        0.0       2.18   
         5                 28.4575             31.25        0.0       1.79   

                    mean_6  median_6  standard_deviation_6  first_quartile_6  \
activity dataset                                                               
bending1 1        0.570583      0.43              0.582915               0.0   
         2        0.571083      0.43              0.601010               0.0   
         3        0.493292      0.43              0.513506               0.0   
         4        0.613521      0.50              0.524317               0.0   
         5        0.383292      0.43              0.389164               0.0   

                  third_quartile_6  
activity dataset                    
bending1 1                    1.30  
         2                    1.30  
         3                    0.94  
         4                    1.00  
         5                    0.50  

[5 rows x 42 columns]

#### iii. Standard Deviation

In [7]:
feature_values = feature_table.to_numpy(dtype=float)
n_instances, n_features = feature_values.shape
n_bootstrap = 10000

# Bootstrap instances (rows) with replacement and estimate each feature's std.
rng = np.random.default_rng(552)
bootstrap_indices = rng.integers(
    0, n_instances, size=(n_bootstrap, n_instances)
)
bootstrap_stds = np.std(feature_values[bootstrap_indices], axis=1, ddof=1)

standard_deviations = np.std(feature_values, axis=0, ddof=1)
ci_lower, ci_upper = np.percentile(bootstrap_stds, [5, 95], axis=0)

std_confidence_intervals = pd.DataFrame(
    {
        "standard_deviation": standard_deviations,
        "90%_CI_lower": ci_lower,
        "90%_CI_upper": ci_upper,
    },
    index=feature_table.columns,
)

assert std_confidence_intervals.shape == (42, 3)
print(f"Bootstrap resamples: {n_bootstrap}")
print(f"Features summarized: {len(std_confidence_intervals)}")
std_confidence_intervals

Bootstrap resamples: 10000
Features summarized: 42


,standard_deviation,90%_CI_lower,90%_CI_upper
minimum_1,9.569975,8.256874,10.741701
maximum_1,4.394362,3.367466,5.284951
mean_1,5.335718,4.702209,5.882310
median_1,5.440054,4.790514,5.992412
standard_deviation_1,1.772153,1.568752,1.948021
first_quartile_1,6.153590,5.573883,6.636206
third_quartile_1,5.138925,4.341653,5.850324
minimum_2,0.000000,0.000000,0.000000
maximum_2,5.062729,4.612622,5.407682
mean_2,1.574164,1.394270,1.706270


#### iv. Select Features

I select **mean, standard deviation, and maximum** for each of the six time series, giving 18 selected features per instance.

- **Mean** summarizes the typical signal level during a recording.
- **Standard deviation** measures within-recording variation, which can distinguish steady activities from more dynamic ones.
- **Maximum** captures peak signal intensity.

Together, these features represent the signal's level, variability, and peak behavior. I prefer them to minimum because the extracted minima for time series 2 and 4 are constant at zero across all 88 recordings, so those minimum features provide no separation between instances.

## 2. ISLR 3.7.4

### (a) Linear Train

We would expect the cubic regression to have a lower or equal training RSS. The linear model is nested within the cubic model, since it is obtained by setting \(\beta_2=\beta_3=0\). Therefore, the cubic model can fit the training data at least as well as the linear model, even when the true relationship is linear.

### (b) Linear Test

We would expect the linear regression model to have a lower test RSS. Since the true relationship is linear, the linear model is correctly specified. The cubic model includes unnecessary \(X^2\) and \(X^3\) terms, which may fit random noise in the training data and lead to worse predictions on new data. This is an expectation; the cubic model could still have a lower RSS on a particular test set.

### (c) Not Linear Train

We would expect the cubic regression to have a lower or equal training RSS. The linear model is nested within the cubic model: the cubic model can reproduce the linear model by setting \(\beta_2=\beta_3=0\). Therefore, the cubic model’s training RSS cannot be higher, regardless of the true relationship. It will be strictly lower if the added terms improve the fit; otherwise, the RSS values will be equal.

### (d) Not Linear Testing

There is not enough information to determine which model will have the lower test RSS. Although the true relationship is nonlinear, it may or may not be well approximated by a cubic function. The cubic model may perform better if it captures the underlying nonlinear pattern, but its additional flexibility may also increase variance and worsen test performance.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 